# Step 2 — The RAG Pipeline

**RAG = Retrieval-Augmented Generation.** A general LLM has never seen Nexora's leave policy, so it guesses. If we **retrieve** the right chunks of the policy and paste them into the prompt, the same model answers correctly and cites its source.

The pipeline has two halves:

| Phase | When | Steps |
|---|---|---|
| **Indexing** | once | load → chunk → embed → store |
| **Answering** | every question | embed question → retrieve top-k → build prompt → generate |

You need **Ollama running** with `llama3.2:3b` for the generation cells.

### Setup
Run this cell first. It lets the notebook import the lab's own code (`config.py`, `rag/`, `agent/`, `llm.py`) from the folder above `notebooks/`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
print("Project folder:", ROOT)

## Part A — Indexing
### A1. Load the document
`parse_file()` reads a `.md`, `.txt` or `.pdf` file into one long string.

In [ ]:
import config
from rag.ingest import parse_file

doc = parse_file(config.DOCS_PATH / "hr_leave_policy.md")
print("Source:", doc["source"])
print("Characters:", len(doc["text"]))
print(doc["text"][:400])

### A2. Chunk it
We cut the text into pieces of about `CHUNK_SIZE` characters, with `CHUNK_OVERLAP` characters shared between neighbours, so a sentence cut at one boundary is still whole in the next chunk.

In [ ]:
from rag.chunkers import chunk_document, chunk_stats

chunks = chunk_document(doc["text"], strategy=config.CHUNK_STRATEGY, size=config.CHUNK_SIZE, overlap=config.CHUNK_OVERLAP)
print(f"strategy={config.CHUNK_STRATEGY}  size={config.CHUNK_SIZE}  overlap={config.CHUNK_OVERLAP}")
print(chunk_stats(chunks) | {"lengths": "..."})

Look at one chunk. Notice the `start`/`end` offsets, the `section` label and how much it overlaps the previous chunk.

👀 The chunk holds the Marriage Leave rules, yet its `section` says *Bereavement Leave*: the label records the heading the chunk **starts** in. Chunk boundaries don't line up with headings (compare the `heading` strategy below).

In [ ]:
marriage = next(c for c in chunks if "of paid Marriage Leave" in c["text"])
{k: v for k, v in marriage.items()}

✏️ **Try it:** compare chunking strategies. `fixed` cuts anywhere, `recursive` prefers paragraph and sentence breaks, `heading` keeps each policy section together.

In [ ]:
for strategy in ["fixed", "recursive", "heading"]:
    cs = chunk_document(doc["text"], strategy=strategy, size=500, overlap=100)
    warnings = sum(len(c["warnings"]) for c in cs)
    print(f"{strategy:<10} {len(cs):>3} chunks   avg {chunk_stats(cs)['avg']:>6} chars   {warnings} warnings")

### A3 + A4. Embed and store in ChromaDB
`build_index()` loads every file in `sample_docs/`, chunks it, embeds each chunk (384 numbers) and stores it in ChromaDB. If the index already exists with the same settings, it skips the work.

In [ ]:
from rag.ingest import build_index, get_collection

build_index()
collection = get_collection()
print("Rows in ChromaDB:", collection.count())

Peek at what ChromaDB really stores for one row: an id, the text, metadata and the embedding.

In [ ]:
row = collection.get(limit=1, include=["documents", "metadatas", "embeddings"])
print("id:       ", row["ids"][0])
print("metadata: ", row["metadatas"][0])
print("text:     ", row["documents"][0][:120], "...")
print("embedding:", [round(float(x), 3) for x in row["embeddings"][0][:6]], "... (384 numbers)")

## Part B — Answering a question
### B1. Retrieve
The question is embedded with the **same** model, and ChromaDB returns the `TOP_K` nearest chunks. `score = 1 − cosine distance` (1.0 = same meaning).

In [ ]:
from rag.retriever import retrieve

QUESTION = "How many days of Sick Leave can be carried forward?"

hits = retrieve(QUESTION, k=config.TOP_K)
for i, h in enumerate(hits, start=1):
    print(f"{i}. score={h['score']:.3f}  [{h['source']} {h['chunk_id']}]  {h['text'].strip()[:110].replace(chr(10), ' ')}...")

### B2. Augment: build the prompt
This is exactly what the LLM will read: a system rule (*answer only from the context*), the numbered chunks, then the question.

In [ ]:
from rag.retriever import build_rag_messages

messages = build_rag_messages(QUESTION, hits)
for m in messages:
    print(f"### {m['role'].upper()}\n{m['content'][:1200]}\n")

### B3. Generate — with vs without RAG
First check the LLM is reachable.

In [ ]:
from llm import check_llm

ok, message = check_llm()
print("LLM ready ✅" if ok else f"LLM NOT ready ❌  {message}")

In [ ]:
from llm import get_llm_response

without_rag = get_llm_response([{"role": "user", "content": QUESTION}])
with_rag = get_llm_response(messages)

print("(a) WITHOUT context:\n", without_rag, "\n")
print("(b) WITH RAG context:\n", with_rag)

### B4. All of it in one call
`rag_answer()` does retrieve → augment → generate and returns the answer with its sources.

✏️ **Try it:** ask a question that is *not* in the policy, e.g. `"What is on the cafeteria menu?"`. The model should say it doesn't know.

In [ ]:
from rag.retriever import rag_answer

result = rag_answer("How many days off can I take for my wedding?")
print("Answer: ", result["answer"])
print("Sources:", result["sources"])